# Instructions for visualizing maps!
#### The objective of this practice is to present some basic visualization methods for preparing maps in hydrologic studies.

Please leverage the code in this script and prepare the maps for the visualization battle!

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
from shapely.geometry import Polygon
import os
import pyproj

In [ ]:
site_id = "01205500"

In [ ]:
# basin shapefile
gdf = gpd.read_file("%s.basin_bound.gpkg"%site_id)

In [ ]:
# domain file
domain_ds = xr.open_dataset("namerica_domain.%s.nc"%site_id)

In [ ]:
# meteorological forcing data
ds_met = xr.open_dataset("ERA5.01205500.regrid.2012.nc")

In [ ]:
# river shapefiles
riv_gdf = gpd.read_file("%s.riv.gpkg"%site_id)

In [ ]:
# elevation maps
elev_ds = xr.open_dataset("elevation.%s.nc"%site_id)

In [ ]:
# projection information
proj1 = ccrs.Mercator()

# 1. The geographical locations of the basin?

In [ ]:
plt.figure(dpi=200)
# set up the projection information for the map
ax = plt.axes(projection=proj1)

# set the range of the basin
ax.set_extent([-78,-70,38,45])

# plot the boundary of your targeted basin
gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax,facecolor='none',
                                                edgecolor='orange',zorder=3)

# set the state boundary
ax.add_feature(cfeature.STATES,lw=0.3,facecolor='none',edgecolor='k')
ax.add_feature(cfeature.OCEAN, facecolor="dodgerblue",edgecolor="none")

# 2. Topograph and River Networks

Topopraph of a basin can provide basic information of the source of water and help us decide the overall flow directions of the rivers.

The data source for the topograph is from USGS. Here is the link to the dataset https://topotools.cr.usgs.gov/gmted_viewer/viewer.htm

In [ ]:
plt.figure(dpi=300)
ax = plt.axes(projection=proj1)

# topography of the basin
elev_ds.elev.plot(x='lon',y='lat',transform=ccrs.PlateCarree(),
                  cmap='terrain',zorder=1,lw=0.4)
gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax,facecolor='none',edgecolor='k', 
                                                linewidth=0.4,zorder=3)


# 3. Side-by-side maps when comparing two datasets
We want to check whether the mountains will receive more precipitation compares to lower elevation regions

In [ ]:
# calculate the mean annual total precipitation
ds_met_pr_annual = ds_met.MTPR.groupby(ds_met.time.dt.year).sum().mean(dim='year')

In [ ]:
# plt.figure(dpi=300)
# ax = plt.axes(projection=proj1)
fig,ax = plt.subplots(1, 2, subplot_kw={'projection':proj1}, 
                      dpi=200, figsize=[10,8])

# left figure
elev_ds.elev.plot(x='lon',y='lat',transform=ccrs.PlateCarree(), ax=ax[0],
                  cmap='terrain',zorder=1,lw=0.4)
gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax[0],facecolor='none',edgecolor='k', 
                                                linewidth=0.4,zorder=3)
# right figure
ds_met_pr_annual.plot(x='lon',y='lat',transform=ccrs.PlateCarree(),ax=ax[1],
                      cmap='Blues',edgecolor='k',zorder=1,lw=0.4)
gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax[1],facecolor='none',edgecolor='k', 
                                                linewidth=0.4,zorder=3)

fig.tight_layout()


# 4. River Networks

In [ ]:
plt.figure(dpi=300)
ax = plt.axes(projection=proj1)

# domain_ds.mask.plot(x='lon',y='lat',transform=ccrs.PlateCarree(),add_colorbar=False,
#                     facecolor='none',edgecolor='silver', zorder=2,lw=0.4)
riv_gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax, color='navy',  
                                                    linewidth=riv_gdf['order']/3, zorder= 3)
gdf.to_crs(pyproj.CRS(proj1.proj4_params)).plot(ax=ax,facecolor='none',edgecolor='orange',zorder=3)
ax.add_feature(cfeature.STATES,edgecolor='silver',lw=0.3)